## Machine Learning Group Project - Transit Reliability and Neighbourhood Equity in Toronto

Training and Predictions


The Data from TTC is joined with the Neighborhood Profiles from the Census, so now we can read it in here

In [23]:
#required imports
import numpy as np
import pandas as pd
import seaborn as sns

# to make this notebook's output stable across runs
np.random.seed(123)

# To plot pretty figures
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12


In [24]:
# in google colab, mount the google drive to access the data
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
pathToGroup = r"/content/drive/My Drive/Colab Notebooks/GroupProject/"

fileName = 'delay_with_neighborhood.xlsx'

dataset = pd.read_excel(pathToGroup+fileName, skiprows= 0, header = 0, index_col = 0)


In the following, you can take a look into the dataset.

In [26]:
dataset.shape

(56206, 60)

In [27]:
# I want to see all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 30)

dataset.head(10)

,Date,Line,Time,Day,Station,Code,Min Delay,Min Gap,Bound,Vehicle,latitude,longitude,AREA_SHORT_CODE,AREA_NAME,AREA_DESC,CLASSIFICATION,CLASSIFICATION_CODE,Total - Age groups of the population - 25% sample data,Median after-tax income of household in 2020 ($),Average after-tax income of household in 2020 ($),Total - Private households by tenure - 25% sample data,Owner,Renter,Total - Citizenship for the population in private households - 25% sample data,Canadian citizens,Not Canadian citizens,Total - Immigrant status and period of immigration for the population in private households - 25% sample data,Non-immigrants,Immigrants,Total - Place of work status for the employed labour force aged 15 years and over - 25% sample data,Worked at home,Worked outside Canada,No fixed workplace address,Usual place of work,Total - Commuting destination for the employed labour force aged 15 years and over with a usual place of work - 25% sample data,Commute within census subdivision (CSD) of residence,Commute to a different census subdivision (CSD) within census division (CD) of residence,Commute to a different census subdivision (CSD) and census division (CD) within province or territory of residence,Commute to a different province or territory,Total - Main mode of commuting for the employed labour force aged 15 years and over with a usual place of work or no fixed workplace address - 25% sample data,"Car, truck or van","Car, truck or van - as a driver","Car, truck or van - as a passenger",Public transit,Walked,Bicycle,Other method,Total - Commuting duration for the employed labour force aged 15 years and over with a usual place of work or no fixed workplace address - 25% sample data,Less than 15 minutes,15 to 29 minutes,30 to 44 minutes,45 to 59 minutes,60 minutes and over,Total - Time leaving for work for the employed labour force aged 15 years and over with a usual place of work or no fixed workplace address - 25% sample data,Between 5 a.m. and 5:59 a.m.,Between 6 a.m. and 6:59 a.m.,Between 7 a.m. and 7:59 a.m.,Between 8 a.m. and 8:59 a.m.,Between 9 a.m. and 11:59 a.m.,Between 12 p.m. and 4:59 a.m.
0,2025-01-01,102 MARKHAM ROAD,02:15,Wednesday,WARDEN STATION,MFESA,20,40,N,3442,43.769610,-79.304280,119,Wexford/Maryvale,Wexford/Maryvale (119),Not an NIA or Emerging Neighbourhood,NaN,28345,74000,85800,10260,6050,4220,28340,23620,4720,28340,12365,14400,12445,3120,25,1805,7490,7490,6240,0,1240,10,9300,6300,5485,815,2290,405,35,265,9300,1815,2840,2395,975,1270,9300,650,1285,1905,1845,1665,1950
1,2025-01-01,65 PARLIAMENT,02:15,Wednesday,KIPLING STATION,MFUS,0,0,NaN,0,43.676448,-79.551377,10,Princess-Rosethorn,Princess-Rosethorn (10),Not an NIA or Emerging Neighbourhood,NaN,11170,133000,164200,3900,3310,590,11170,10460,710,11170,7615,3400,5215,2260,15,490,2450,2450,1770,0,675,0,2940,2435,2230,205,290,95,15,110,2940,640,1085,705,270,240,2940,105,360,730,865,625,260
2,2025-01-01,64 MAIN,02:40,Wednesday,BROADVIEW STATION,MFUI,0,0,NaN,8546,43.674153,-79.355380,68,North Riverdale,North Riverdale (68),Not an NIA or Emerging Neighbourhood,NaN,11290,93000,127900,4905,2855,2045,11290,10680,610,11290,8395,2720,5760,3145,25,395,2205,2205,2010,0,175,20,2600,1155,1020,130,615,465,270,90,2600,460,1010,725,215,190,2600,40,280,690,770,600,225
3,2025-01-01,100 FLEMINGDON PARK,02:43,Wednesday,OVERLEA AND THORNCLIFF,MFSAN,17,32,N,8693,43.706435,-79.345772,55,Thorncliffe Park,Thorncliffe Park (55),Neighbourhood Improvement Area,NIA,20400,62400,68500,7065,990,6070,20400,15215,5185,20400,6420,12905,7140,2130,50,1330,3630,3630,3070,0,550,10,4960,2880,2590,285,1465,405,40,165,4960,925,1285,1380,715,655,4960,245,640,985,920,940,1230
4,2025-01-01,34 EGLINTON EAST,03:05,Wednesday,EGLINTON AND DON MILLS,MFUI,20,40,W,8801,43.720512,-79.338919,42,Banbury-Don Mills,Banbury-Don Mills (42),Not an NIA or Emerging Neighbourhood,NaN,27155,78000,109700,12215,7295,4915,27150,23375,3780,27150,12640,13675,12325,5305,190,980,5850,5855,4940,0,895,15,6835,5015,4540,480,1145,335,65,270,6835,11